# 电影圈数据

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import scipy
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
from networkx.algorithms import bipartite

In [2]:
import sys
sys.path.append("..")

# 数据处理

## 原始数据

In [3]:
df_raw = pd.read_csv("dwd_cast_works.csv")
df_raw.head()

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,k_genres,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,喜剧/动作,无评分,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,爱情,无评分,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,剧情,无评分,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓
4,1013885,阿美莉嘉·奥利沃,女,10727641,演员,是,21,10727641,碟中谍5：神秘国度,电影,...,动作/惊悚/冒险,有评分,7.8,292193,292193.0,1510.0,21455331,2027819,5,Turandot


## 数据筛选
过滤条件
1. 电影
2. 有评分
3. 地区不含其他合拍

In [4]:
# df_movie = df_raw.loc[(df_raw['k_type'] == '电影')
#                       & (df_raw['k_region'].str.contains('中国'))].copy()
df_movie = df_raw.loc[(df_raw['k_region'] != '其他合拍')].copy()
# k_cast_id转为str
df_movie['k_cast_id'] = df_movie['k_cast_id'].apply(lambda x: str(x))
# 仅保留演员，导演，编剧数据
df_movie = df_movie[df_movie['k_role'].isin(['演员', '导演', '编剧'])]
df_movie = df_movie.reset_index(drop=True).copy(deep=True)
# 添加新的movie_id_m列
df_movie['movie_id_m'] = df_movie['k_movie_id'].apply(lambda x: 'm' + str(x))
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,无评分,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN,m21534411
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,无评分,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN,m21236655
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,无评分,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN,m21202445
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,无评分,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓,m21133377
4,1003424,蒋君超,男,10777680,演员,是,2,10777680,影坛风月,电影,...,无评分,NaN,暂无评分,NaN,NaN,21555409,2006897,6,NaN,m21555409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494104,34951835,李卿,男,36744438,演员,是,17,36744438,书卷一梦,电视剧,...,无评分,NaN,暂无评分,NaN,NaN,73488925,69903719,608471,断山虎,m73488925
494105,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,有评分,5.7,2003,2003.0,107.0,14107525,70830237,608472,NaN,m14107525
494106,1314059,徐小凤,女,26901136,演员,否,999,26901136,天龙酒店,电影,...,无评分,NaN,暂无评分,NaN,NaN,53802321,2628167,608473,NaN,m53802321
494107,1385616,马英智,男,34925531,演员,是,3,34925531,小戏骨：红色娘子军,电视剧,...,无评分,NaN,暂无评分,NaN,NaN,69851111,2771281,608474,南霸天,m69851111


In [5]:
df_movie[df_movie['k_title'] == '让子弹飞']

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,is_rating,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m
25771,30276936,陈磊,男,3742360,演员,否,999,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,60553921,31108,受辱民女夫,m7484769
57316,27504347,杨奇雨,男,3742360,演员,是,21,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,55008743,69834,胡百,m7484769
94695,27250655,周润发,男,3742360,演员,是,3,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,54501359,115968,黄四郎 / 杨万楼,m7484769
106624,27238280,胡军,男,3742360,演员,是,14,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,54476609,130630,假张麻子,m7484769
112259,27504256,述平,男,3742360,编剧,是,2,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,55008561,137532,NaN,m7484769
128910,27481219,冯小刚,男,3742360,演员,是,13,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,54962487,158077,汤师爷,m7484769
131480,30276937,沙瑀,男,3742360,演员,否,999,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,60553923,161321,鹅人,m7484769
156272,27504255,朱苏进,男,3742360,编剧,是,1,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,55008559,191744,NaN,m7484769
172038,27495331,郭俊立,男,3742360,编剧,是,4,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,54990711,211396,NaN,m7484769
172900,27232187,张默,男,3742360,演员,是,6,3742360,让子弹飞,电影,...,有评分,9.0,1571537,1571537.0,3761.0,7484769,54464423,212475,老六,m7484769


## 影人职责合并

In [6]:
# 合并影人职责
df_cast = df_movie[['k_cast_id', 'cast_name', 'k_role']].drop_duplicates()
df_cast_agg = df_cast.groupby(['k_cast_id', 'cast_name'])['k_role'].apply(lambda x: '/'.join(sorted(x.unique()))).reset_index()
df_cast_agg

,k_cast_id,cast_name,k_role
0,2000437,李香凝,演员
1,2000457,吉娜·卡拉诺,演员
2,2000515,理查德·格里克,演员
3,2000941,司汗,演员
4,2001011,凯文·格劳特,导演
...,...,...,...
61324,76124367,张若伊,编剧
61325,76135081,何田田,演员
61326,76349227,方誉媛,演员
61327,76350573,景德军,导演


In [7]:
df_movie['cast_role_agg'] = df_movie['k_cast_id'].map(
    df_cast_agg.set_index('k_cast_id')['k_role']
)
df_movie

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
0,1007816,凌玲,女,10767181,演员,是,5,10767181,妙手千金,电影,...,NaN,暂无评分,NaN,NaN,21534411,2015681,1,NaN,m21534411,演员
1,1003352,金焰,男,10618303,演员,是,1,10618303,失去的爱情,电影,...,NaN,暂无评分,NaN,NaN,21236655,2006753,2,NaN,m21236655,演员
2,1003352,金焰,男,10601198,演员,是,1,10601198,人道,电影,...,NaN,暂无评分,NaN,NaN,21202445,2006753,3,NaN,m21202445,演员
3,1003352,金焰,男,10566664,演员,是,2,10566664,母亲,电影,...,NaN,暂无评分,NaN,NaN,21133377,2006753,4,老邓,m21133377,演员
4,1003424,蒋君超,男,10777680,演员,是,2,10777680,影坛风月,电影,...,NaN,暂无评分,NaN,NaN,21555409,2006897,6,NaN,m21555409,导演/演员
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494104,34951835,李卿,男,36744438,演员,是,17,36744438,书卷一梦,电视剧,...,NaN,暂无评分,NaN,NaN,73488925,69903719,608471,断山虎,m73488925,演员
494105,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,5.7,2003,2003.0,107.0,14107525,70830237,608472,NaN,m14107525,演员
494106,1314059,徐小凤,女,26901136,演员,否,999,26901136,天龙酒店,电影,...,NaN,暂无评分,NaN,NaN,53802321,2628167,608473,NaN,m53802321,演员
494107,1385616,马英智,男,34925531,演员,是,3,34925531,小戏骨：红色娘子军,电视剧,...,NaN,暂无评分,NaN,NaN,69851111,2771281,608474,南霸天,m69851111,演员


In [8]:
df_movie[df_movie['cast_name'] == '张艺谋']

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_num,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg
729,27260166,张艺谋,男,1292365,导演,是,1,1292365,活着,电影,...,9.3,678477,678477.0,2512.0,2584779,54520381,762,NaN,m2584779,导演/演员/编剧
3939,27260166,张艺谋,男,1294963,导演,是,1,1294963,一个都不能少,电影,...,7.7,120767,120767.0,964.0,2589975,54520381,4254,NaN,m2589975,导演/演员/编剧
7799,27260166,张艺谋,男,35294995,演员,是,40,35294995,我和我的父辈,电影,...,6.9,175739,175739.0,1101.0,70590039,54520381,9029,电视台台长,m70590039,导演/演员/编剧
15527,27260166,张艺谋,男,1294007,导演,是,1,1294007,我的父亲母亲,电影,...,8.2,116620,116620.0,978.0,2588063,54520381,18493,NaN,m2588063,导演/演员/编剧
27119,27260166,张艺谋,男,1308070,演员,是,1,1308070,老井,电影,...,8.0,16354,16354.0,362.0,2616189,54520381,32761,孙旺泉,m2616189,导演/演员/编剧
38566,27260166,张艺谋,男,26694491,导演,是,1,26694491,大红灯笼高高挂,电影,...,8.8,3372,3372.0,172.0,53389031,54520381,46781,NaN,m53389031,导演/演员/编剧
42327,27260166,张艺谋,男,1297879,演员,否,999,1297879,大阅兵,电影,...,6.8,1946,1946.0,115.0,2595807,54520381,51418,军官,m2595807,导演/演员/编剧
107457,27260166,张艺谋,男,35215390,编剧,是,2,35215390,狙击手,电影,...,7.7,340264,340264.0,1619.0,70430829,54520381,131623,NaN,m70430829,导演/演员/编剧
107458,27260166,张艺谋,男,35215390,导演,是,1,35215390,狙击手,电影,...,7.7,340264,340264.0,1619.0,70430829,54520381,131624,NaN,m70430829,导演/演员/编剧
107469,27260166,张艺谋,男,33447633,导演,是,1,33447633,坚如磐石,电影,...,6.0,344998,344998.0,1439.0,66895315,54520381,131636,NaN,m66895315,导演/演员/编剧


## 主演数据

In [12]:
# 每部电影的主要演员，is_main_cast==‘是’， k_role==‘演员’, index小于=5
# 为每行添加一个main_cast列，按k_movie_id分组，按is_main_cast==‘是’，k_role=='演员'降序，index升序排序后，取前5个演员的cast_name连接起来
# 1. 筛选并排序：只保留“是演员”且“是主演”的行，按 index 升序
cast_sorted = df_movie[(df_movie['is_main_cast'] == '是') & (df_movie['k_role'] == '演员')] \
    .sort_values(by=['k_movie_id', 'index'], ascending=[True, True])

# 2. 分组聚合：取每个电影前5名，用逗号连接
main_cast_series = cast_sorted.groupby('k_movie_id')['cast_name'] \
    .apply(lambda x: ', '.join(x.head(5)))

# 3. 映射回原表：使用 map 匹配 k_movie_id，确保索引正确
df_movie['main_cast'] = df_movie['k_movie_id'].map(main_cast_series)
df_movie[df_movie['k_title'] == '让子弹飞']

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg,main_cast
25771,30276936,陈磊,男,3742360,演员,否,999,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,60553921,31108,受辱民女夫,m7484769,演员,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
57316,27504347,杨奇雨,男,3742360,演员,是,21,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,55008743,69834,胡百,m7484769,演员,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
94695,27250655,周润发,男,3742360,演员,是,3,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,54501359,115968,黄四郎 / 杨万楼,m7484769,演员,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
106624,27238280,胡军,男,3742360,演员,是,14,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,54476609,130630,假张麻子,m7484769,导演/演员,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
112259,27504256,述平,男,3742360,编剧,是,2,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,55008561,137532,NaN,m7484769,演员/编剧,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
128910,27481219,冯小刚,男,3742360,演员,是,13,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,54962487,158077,汤师爷,m7484769,导演/演员/编剧,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
131480,30276937,沙瑀,男,3742360,演员,否,999,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,60553923,161321,鹅人,m7484769,演员,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
156272,27504255,朱苏进,男,3742360,编剧,是,1,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,55008559,191744,NaN,m7484769,编剧,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
172038,27495331,郭俊立,男,3742360,编剧,是,4,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,54990711,211396,NaN,m7484769,导演/编剧,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"
172900,27232187,张默,男,3742360,演员,是,6,3742360,让子弹飞,电影,...,1571537,1571537.0,3761.0,7484769,54464423,212475,老六,m7484769,演员,"姜文, 葛优, 周润发, 刘嘉玲, 陈坤"


## 有评分数据

In [13]:
df_movie_rated = df_movie[df_movie['is_rating'] == '有评分'].copy()
df_movie_rated = df_movie_rated.reset_index(drop=True).copy(deep=True)
df_movie_rated

,cast_id,cast_name,gender,movie_id,k_role,is_main_cast,index,id,k_title,k_type,...,rating_people,k_rating_people,rating_index,k_movie_id,k_cast_id,uid,portray,movie_id_m,cast_role_agg,main_cast
0,1017182,陈燕燕,女,10771202,演员,是,1,10771202,深闺疑云,电影,...,92,92.0,27.0,21542453,2034413,54,赵兰,m21542453,演员,"陈燕燕, 谢添"
1,1043845,刘志荣,男,10581318,演员,是,11,10581318,四大家族之龙虎兄弟,电影,...,2290,2290.0,134.0,21162685,2087739,75,NaN,m21162685,导演/演员/编剧,"郑则仕, 吕良伟, 利智, 曾江, 黄光亮"
2,1050373,茅瑛,女,10573512,演员,是,1,10573512,鬼娘子,电影,...,235,235.0,37.0,21147073,2100795,79,NaN,m21147073,演员,"茅瑛, 林威"
3,1124360,利智,女,10581318,演员,是,3,10581318,四大家族之龙虎兄弟,电影,...,2290,2290.0,134.0,21162685,2248769,94,NaN,m21162685,演员,"郑则仕, 吕良伟, 利智, 曾江, 黄光亮"
4,1314471,王挺,男,10531190,演员,是,1,10531190,惊天动地,电视剧,...,326,326.0,53.0,21062429,2628991,147,NaN,m21062429,导演/演员/编剧,"王挺, 李易祥, 王双宝"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283251,1354509,林泳淘,女,26984991,演员,否,999,26984991,婚姻合伙人,电视剧,...,1259,1259.0,85.0,53970031,2709067,608462,邓秀玉,m53970031,演员,"黄淑仪, 杨明, 高海宁, 黄德斌, 陆永"
283252,27496534,刘芊蒂,女,2337597,演员,是,2,2337597,操行零分,电影,...,211,211.0,38.0,4675243,54993117,608466,NaN,m4675243,演员,"何启南, 刘芊蒂"
283253,27565771,董亚春,女,2270516,导演,是,2,2270516,八月一日,电影,...,1123,1123.0,85.0,4541081,55131591,608467,NaN,m4541081,导演,"刘劲, 侯勇, 吕良伟"
283254,35415094,张可,女,7053738,演员,否,999,7053738,樱桃,电视剧,...,2003,2003.0,107.0,14107525,70830237,608472,NaN,m14107525,演员,"沈春阳, 宋小宝, 小沈阳, 赵本山"


## main

In [ ]:
def works_data_process(df_raw):
    df_movie = df_raw.loc[(df_raw['k_region'] != '其他合拍')].copy()
    # k_cast_id转为str
    df_movie['k_cast_id'] = df_movie['k_cast_id'].apply(lambda x: str(x))
    # 仅保留演员，导演，编剧数据
    df_movie = df_movie[df_movie['k_role'].isin(['演员', '导演', '编剧'])]
    df_movie = df_movie.reset_index(drop=True).copy(deep=True)
    # 添加新的movie_id_m列
    df_movie['movie_id_m'] = df_movie['k_movie_id'].apply(
        lambda x: 'm' + str(x))
    # 合并影人职责
    df_cast = df_movie[['k_cast_id', 'cast_name', 'k_role']].drop_duplicates()
    df_cast_agg = df_cast.groupby([
        'k_cast_id', 'cast_name'
    ])['k_role'].apply(lambda x: '/'.join(sorted(x.unique()))).reset_index()
    df_movie['cast_role_agg'] = df_movie['k_cast_id'].map(
        df_cast_agg.set_index('k_cast_id')['k_role'])
    # 每部电影的主要演员，is_main_cast==‘是’， k_role==‘演员’, index小于=5
    # 为每行添加一个main_cast列，按k_movie_id分组，按is_main_cast==‘是’，k_role=='演员'降序，index升序排序后，取前5个演员的cast_name连接起来
    # 1. 筛选并排序：只保留“是演员”且“是主演”的行，按 index 升序
    cast_sorted = df_movie[(df_movie['is_main_cast'] == '是')
                           & (df_movie['k_role'] == '演员')].sort_values(
                               by=['k_movie_id', 'index'],
                               ascending=[True, True])

    # 2. 分组聚合：取每个电影前5名，用逗号连接
    main_cast_series = cast_sorted.groupby('k_movie_id')['cast_name'].apply(
        lambda x: ', '.join(x.head(5)))

    # 3. 映射回原表：使用 map 匹配 k_movie_id，确保索引正确
    df_movie['main_cast'] = df_movie['k_movie_id'].map(main_cast_series)

    # 将int64类型的列转换为普通int类型
    # int64_cols = df_movie.select_dtypes(include=['int64']).columns
    # for col in int64_cols:
    #     df_movie[col] = df_movie[col].astype(int)
    return df_movie


## 保存为csv

In [18]:
file_name = "zwyg"
df_raw = pd.read_csv(f"../data/{file_name}.csv")
df_movie = works_data_process(df_raw)
df_movie_rated = df_movie[df_movie['is_rating'] == '有评分'].copy()
df_movie_rated = df_movie_rated.reset_index(drop=True).copy(deep=True)
df_movie.to_csv(f"../data/{file_name}_processed.csv", index=False)
df_movie_rated.to_csv(f"../data/{file_name}_rated_processed.csv", index=False)

# 数据统计

## 年度数据

In [ ]:
# 按年统计每年的电影数量
df_count_by_year = df_movie_rated.groupby('k_movie_year')['movie_id_m'].nunique().reset_index()
df_count_by_year.columns = ['year', 'movie_count']
df_count_by_year = df_count_by_year.sort_values('year')
df_count_by_year

In [ ]:
# 绘图
plt.figure(figsize=(10, 6))
plt.plot(df_count_by_year['year'], df_count_by_year['movie_count'], marker='o')
plt.title('年度电影数量统计')
plt.xlabel('年份')
plt.ylabel('电影数量')
plt.xticks(df_count_by_year['year'], rotation=45)
# plt.tight_layout()
plt.show()